In [1]:
import os
import glob
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score

# グラフのスタイルとフォントサイズを設定
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'font.size': 12})

In [3]:
# --- ユーザー設定 ---
# データが格納されているメインディレクトリのパスを指定してください
# (Windowsの場合、パスの前に r を付けるか、バックスラッシュを2つにしてください)
base_dir = r"E:\permian_basin\data_5"

# 背景データとテストデータのディレクトリ
bg_dir = os.path.join(base_dir, "bg", "normalized")
test_dir = os.path.join(base_dir, "test", "normalized")

# 背景CH4濃度 (ppm)
BG_CH4_CONC = 1.8

# C指数を計算する波長範囲 (nm)
WAVELENGTH_MIN = 2248
WAVELENGTH_MAX = 2298

# C指数を計算する際のスケール係数 (論文参照)
C_SCALE_FACTOR = 100000

print(f"背景データディレクトリ: {bg_dir}")
print(f"テストデータディレクトリ: {test_dir}")

背景データディレクトリ: E:\permian_basin\data_5\bg\normalized
テストデータディレクトリ: E:\permian_basin\data_5\test\normalized


In [4]:
results = []

# testディレクトリ内の全CSVファイルを取得
test_files = glob.glob(os.path.join(test_dir, "*.csv"))

print(f"処理対象のファイル数: {len(test_files)}件")

for test_filepath in test_files:
    filename = os.path.basename(test_filepath)
    
    # ファイル名から反射率とCH4濃度を抽出
    match = re.search(r"test_surref_([\d.]+)_ch4_([\d.]+)_scan\.csv", filename)
    if not match:
        continue
        
    surref = float(match.group(1))
    ch4_conc = float(match.group(2))
    
    # 反射率が0の場合は計算できないためスキップ
    if surref == 0.0:
        continue

    try:
        # 対応する背景データファイルのパスを作成
        bg_filename = f"bg_surref_{surref:.2f}_scan.csv"
        bg_filepath = os.path.join(bg_dir, bg_filename)
        
        # データの読み込み
        df_test = pd.read_csv(test_filepath)
        df_bg = pd.read_csv(bg_filepath)
        conversion_factor = 0.01
        df_test['radiance'] = df_test['radiance'] * conversion_factor
        df_bg['radiance'] = df_bg['radiance'] * conversion_factor


        # 波長範囲でデータをフィルタリング
        df_test_filtered = df_test[(df_test['wave_nm'] >= WAVELENGTH_MIN) & (df_test['wave_nm'] <= WAVELENGTH_MAX)]
        df_bg_filtered = df_bg[(df_bg['wave_nm'] >= WAVELENGTH_MIN) & (df_bg['wave_nm'] <= WAVELENGTH_MAX)]
        
        # スペクトル残差を計算
        l_res = df_bg_filtered['radiance'].values - df_test_filtered['radiance'].values
        
        # 平均残差を計算
        mean_residual = np.mean(l_res)
        
        # C指数を計算 (論文の式(3)参照)
        c_index = (mean_residual / surref) * C_SCALE_FACTOR
        
        results.append({
            'ch4_conc': ch4_conc,
            'surref': surref,
            'c_index': c_index
        })

    except FileNotFoundError:
        print(f"警告: 対応する背景ファイルが見つかりません: {bg_filename}")
    except Exception as e:
        print(f"エラー: ファイル {filename} の処理中に問題が発生しました: {e}")

# 結果をPandas DataFrameに変換
results_df = pd.DataFrame(results)

print(f"\n計算完了。{len(results_df)}件のデータを処理しました。")
results_df.head()

処理対象のファイル数: 707件

計算完了。700件のデータを処理しました。


,ch4_conc,surref,c_index
0,1.9,0.005,887.827633
1,2.1,0.005,891.035530
2,2.3,0.005,894.186143
3,2.8,0.005,901.846239
4,3.8,0.005,916.411307


In [5]:
# フィッティング対象のデータを抽出
x_data = results_df['c_index']
y_data = results_df['ch4_conc']

# 2次の多項式でフィッティング
# y = a*x^2 + b*x + c
coeffs = np.polyfit(x_data, y_data, 2)
a, b, c = coeffs

# フィッティングされた多項式を作成
polynomial = np.poly1d(coeffs)

# R^2 (決定係数) を計算
y_pred = polynomial(x_data)
r2 = r2_score(y_data, y_pred)

print("--- フィッティング結果 ---")
print(f"導出された二次式の係数 (a, b, c): {a:.6f}, {b:.6f}, {c:.6f}")
print(f"\nCH4濃度を求める式:")
print(f"  CH4 (ppm) = {a:.6f} * C^2 + {b:.6f} * C + {c:.6f}")
print(f"\n決定係数 (R^2): {r2:.6f}")

# C=0のときのCH4濃度（切片）が背景濃度に近いか確認
print(f"C=0 のときの推定CH4濃度（切片）: {c:.4f} ppm (設定した背景濃度: {BG_CH4_CONC} ppm)")

--- フィッティング結果 ---
導出された二次式の係数 (a, b, c): -0.000012, 0.012070, 3.274430

CH4濃度を求める式:
  CH4 (ppm) = -0.000012 * C^2 + 0.012070 * C + 3.274430

決定係数 (R^2): 0.130373
C=0 のときの推定CH4濃度（切片）: 3.2744 ppm (設定した背景濃度: 1.8 ppm)
